# Notebook 03 — AlphaFold2: With vs. Without MSA

**Learning objectives:**
- Understand what an MSA (Multiple Sequence Alignment) is and why it matters
- See how prediction confidence changes with vs. without evolutionary information
- Understand why BindCraft's i_pTM is especially meaningful for designed sequences

**Time:** ~30 minutes

**Companion wiki page:** [4.4 MSAs and Evolution](https://rucha1796.github.io/internship-bindcraft-2026/m4_04_msa/)

---
> **GPU required.** Runtime → Change runtime type → T4 GPU

In [ ]:
# Install (skip if already done in NB02)
!pip install -q "colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"
!pip install -q py3Dmol matplotlib
print("✓ Ready")

## Part 1 — PD-L1 with full MSA (normal ColabFold)

PD-L1 is a well-studied human protein. ColabFold will find thousands of homologs in its databases, giving AF2 rich evolutionary co-variation signals.

In [ ]:
import os, json, glob, numpy as np, matplotlib.pyplot as plt

PDL1_SEQUENCE = "MIFALLAALAVFPVSQAMQTEVDGTQNLTICRFPVEKQLDLAALIVYWEMEDKNIIQFVHGEEDLKVQHSSYRQRARLLKDQLSLGNAALQITDVKLQDAGVYRCMISYGGADYKRITVKVNAPYNKINQRILVDPQRLYLVHNQKLCILHQKFTTL"

os.makedirs("nb03_full_msa", exist_ok=True)
with open("PDL1.fasta", "w") as f:
    f.write(">PDL1\n" + PDL1_SEQUENCE + "\n")

print("Running PD-L1 with FULL MSA (normal ColabFold)...")
!colabfold_batch PDL1.fasta nb03_full_msa/ \
    --num-recycle 1 --num-models 1 \
    --model-type alphafold2_ptm \
    2>&1 | grep -E '(pLDDT|pTM|Done|Error)' || true
print("\n✓ Full MSA run complete")

## Part 2 — Designed sequence (single-sequence, no MSA)

Here we use a short example of a *designed* helical peptide. Since it doesn't exist in any database, ColabFold will find no homologs — AF2 must rely on its learned physics knowledge only.

In [ ]:
# A typical BindCraft-style helical sequence: designed, no natural homologs
# (This is an example; your BindCraft run will produce real ones)
DESIGNED_SEQUENCE = "MAEIAALEQKLQALEKKLQALEQKLEALEKKLQALEQKLQALEKKLQALEQ"

os.makedirs("nb03_single_seq", exist_ok=True)
with open("designed_binder.fasta", "w") as f:
    f.write(">designed_helical_binder\n" + DESIGNED_SEQUENCE + "\n")

print(f"Designed sequence length: {len(DESIGNED_SEQUENCE)} residues")
print("Running with single-sequence mode (no MSA search)...")

!colabfold_batch designed_binder.fasta nb03_single_seq/ \
    --num-recycle 1 --num-models 1 \
    --model-type alphafold2_ptm \
    --msa-mode single_sequence \
    2>&1 | grep -E '(pLDDT|pTM|Done|Error)' || true

print("\n✓ Single-sequence run complete")

## Cell — Compare pLDDT profiles side by side

In [ ]:
def load_scores(folder):
    score_files = sorted(glob.glob(f"{folder}/*scores*.json"))
    if not score_files:
        return None
    with open(score_files[0]) as f:
        return json.load(f)

pdl1_scores = load_scores("nb03_full_msa")
designed_scores = load_scores("nb03_single_seq")

if pdl1_scores and designed_scores:
    pdl1_plddt = pdl1_scores.get("plddt", [])
    designed_plddt = designed_scores.get("plddt", [])
    pdl1_ptm = pdl1_scores.get("ptm", 0)
    designed_ptm = designed_scores.get("ptm", 0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.bar(range(1, len(pdl1_plddt)+1), pdl1_plddt, color='#3c5b6f', alpha=0.8)
    ax1.axhline(70, color='gray', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, 100)
    ax1.set_title(f"PD-L1 (full MSA)\npTM = {pdl1_ptm:.3f} | mean pLDDT = {np.mean(pdl1_plddt):.1f}", fontsize=11)
    ax1.set_xlabel("Residue")
    ax1.set_ylabel("pLDDT")

    ax2.bar(range(1, len(designed_plddt)+1), designed_plddt, color='#B76E79', alpha=0.8)
    ax2.axhline(70, color='gray', linestyle='--', alpha=0.5)
    ax2.set_ylim(0, 100)
    ax2.set_title(f"Designed helical sequence (no MSA)\npTM = {designed_ptm:.3f} | mean pLDDT = {np.mean(designed_plddt):.1f}", fontsize=11)
    ax2.set_xlabel("Residue")
    ax2.set_ylabel("pLDDT")

    plt.suptitle("With MSA vs. Without MSA: Confidence Comparison", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig("msa_comparison.png", dpi=150, bbox_inches='tight')
    plt.show()

    print("\n--- Summary ---")
    print(f"PD-L1 (full MSA):          pTM = {pdl1_ptm:.3f} | mean pLDDT = {np.mean(pdl1_plddt):.1f}")
    print(f"Designed (no MSA):         pTM = {designed_ptm:.3f} | mean pLDDT = {np.mean(designed_plddt):.1f}")
    print()
    print("Interpretation:")
    print("- PD-L1 gets high confidence because AF2 has seen thousands of homologs")
    print("- The designed sequence gets lower confidence because AF2 has no evolutionary data")
    print("- This is why high i_pTM for a designed binder is meaningful: it passes")
    print("  even without evolutionary data, purely from AF2's learned structure knowledge")

## Cell — Visualize both predicted structures

In [ ]:
import py3Dmol

pdl1_pdb = sorted(glob.glob("nb03_full_msa/*.pdb"))
designed_pdb = sorted(glob.glob("nb03_single_seq/*.pdb"))

if pdl1_pdb:
    with open(pdl1_pdb[0]) as f: pdl1_str = f.read()
    with open(designed_pdb[0]) as f: designed_str = f.read()

    print("PD-L1 (blue) — with full MSA")
    v1 = py3Dmol.view(width=700, height=400)
    v1.addModel(pdl1_str, "pdb")
    v1.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
    v1.zoomTo()
    v1.setBackgroundColor("white")
    v1.show()

    print("\nDesigned helical binder (rose) — no MSA")
    v2 = py3Dmol.view(width=700, height=400)
    v2.addModel(designed_str, "pdb")
    v2.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
    v2.zoomTo()
    v2.setBackgroundColor("white")
    v2.show()
    
    print("Color: blue = high confidence, red = low confidence")

## Questions to answer

**Q1:** What is the pTM score for PD-L1 (full MSA)? For the designed sequence (no MSA)? What's the difference?

_Your answer:_

**Q2:** Looking at the pLDDT plots, which protein has more high-confidence residues? Why does this make sense?

_Your answer:_

**Q3:** Why does BindCraft use AF2 in single-sequence mode for designed binders? If AF2 gives lower confidence for designed binders, why is the i_pTM filter still useful?

_Your answer:_

**Q4:** The designed sequence here is a simple repeated helix (`MAEIAALEQKLQ...`). Does it fold confidently as a helix? What does the structure look like?

_Your answer:_

---
**Next:** Notebook 04 — Running BindCraft to design binders against PD-L1